In [1]:
data_abs_path = "/home/sxi219/RAI-Account/MSR_data_cleaned.csv"


In [2]:
import pandas as pd

df = pd.read_csv(data_abs_path, nrows=1000)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 36 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Unnamed: 0                    1000 non-null   int64  
 1   Access Gained                 252 non-null    object 
 2   Attack Origin                 1000 non-null   object 
 3   Authentication Required       1000 non-null   object 
 4   Availability                  965 non-null    object 
 5   CVE ID                        1000 non-null   object 
 6   CVE Page                      1000 non-null   object 
 7   CWE ID                        913 non-null    object 
 8   Complexity                    1000 non-null   object 
 9   Confidentiality               621 non-null    object 
 10  Integrity                     557 non-null    object 
 11  Known Exploits                0 non-null      float64
 12  Publish Date                  1000 non-null   object 
 13  Scor

In [4]:
print(df.columns)

Index(['Unnamed: 0', 'Access Gained', 'Attack Origin',
       'Authentication Required', 'Availability', 'CVE ID', 'CVE Page',
       'CWE ID', 'Complexity', 'Confidentiality', 'Integrity',
       'Known Exploits', 'Publish Date', 'Score', 'Summary', 'Update Date',
       'Vulnerability Classification', 'add_lines', 'codeLink', 'commit_id',
       'commit_message', 'del_lines', 'file_name', 'files_changed',
       'func_after', 'func_before', 'lang', 'lines_after', 'lines_before',
       'parentID', 'patch', 'project', 'project_after', 'project_before',
       'vul', 'vul_func_with_fix'],
      dtype='object')


In [5]:
df_code_after = df['func_after']
df_code_lang = df['lang'].str.lower()

In [6]:
df_code_after.info()

<class 'pandas.core.series.Series'>
RangeIndex: 1000 entries, 0 to 999
Series name: func_after
Non-Null Count  Dtype 
--------------  ----- 
1000 non-null   object
dtypes: object(1)
memory usage: 7.9+ KB


# IPAG Builder Experiment

In [7]:
from src.graph.ipag_builder import IPAGBuilder
from src.graph.build_language import LanguageBuilder

langs = set(df_code_lang.str.lower())
lang_map = LanguageBuilder(langs)
ipag = IPAGBuilder(source = df_code_after, language = df_code_lang.str.lower(), lang_map=lang_map.build())
ipag.build()
df_ipag = ipag.get_ipag_dataframe()


Building ASTs for all code snippets
Processed 100/1000 snippets...
Processed 200/1000 snippets...
Processed 300/1000 snippets...
Processed 400/1000 snippets...
Processed 500/1000 snippets...
Processed 600/1000 snippets...
Processed 700/1000 snippets...
Processed 800/1000 snippets...
Processed 900/1000 snippets...
Processed 1000/1000 snippets...
Finished building ASTs
Successful: 1000, Failed: 0, Total: 1000
IPAG Construction Complete
Total snippets: 1000
Average nodes per snippet: 261.3
Average edges per snippet: 260.3


## Building Node features for GIN training

In [8]:
from src.graph.features import BuildNodeFeatures

builder = BuildNodeFeatures(device='cuda')

all_ipag_nodes = df_ipag['ipag_nodes'].tolist()
all_ipag_edges = df_ipag['ipag_edges'].tolist()

features_batch = builder.process_ipag_batch(all_ipag_nodes, all_ipag_edges)


Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BuildNodeFeatures initialized on device: cuda

Processing IPAG 1/1000...
Extracting features for 119 nodes...
Feature extraction complete. Feature dimension: 776

Processing IPAG 2/1000...
Extracting features for 131 nodes...
Feature extraction complete. Feature dimension: 776

Processing IPAG 3/1000...
Extracting features for 302 nodes...
Feature extraction complete. Feature dimension: 776

Processing IPAG 4/1000...
Extracting features for 789 nodes...
Processed 789/789 nodes...
Feature extraction complete. Feature dimension: 776

Processing IPAG 5/1000...
Extracting features for 202 nodes...
Feature extraction complete. Feature dimension: 776

Processing IPAG 6/1000...
Extracting features for 183 nodes...
Feature extraction complete. Feature dimension: 776

Processing IPAG 7/1000...
Extracting features for 317 nodes...
Feature extraction complete. Feature dimension: 776

Processing IPAG 8/1000...
Extracting features for 722 nodes...
Feature extraction complete. Feature dimension: 776

In [ ]:
builder.save_features(features=features_batch, filepath="/home/sxi219/RAI-Account/multi-stage-GNN-code-security/IPAG-GIN-CFEExplainer/data/processed/1000.pkl")